# House vote graph
Member -- bill edge = the member voted **Yea** on the bill's final House passage vote. Bills include those that passed and those that failed (`outcome` tells which).

Build the data first: `uv run python build_graph.py --congress 118`. Change `CONGRESS` below to look at another one you have built.

In [ ]:
import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import comb

CONGRESS = 118
G = nx.read_graphml(f"data/{CONGRESS}/graph.graphml")
members = [n for n, d in G.nodes(data=True) if d["kind"] == "member"]
bills = [n for n, d in G.nodes(data=True) if d["kind"] == "bill"]

MEMBER_COLOR = "#1f77b4"
PASSED_COLOR, FAILED_COLOR = "#2ca02c", "#d62728"
PARTY_COLOR = {"D": "#1f77b4", "R": "#d62728", "Other": "#7f7f7f"}

## 1. Basic stats

In [ ]:
bill_df = pd.DataFrame([{"bill": b, **G.nodes[b]} for b in bills]).set_index("bill")
mem_df = pd.DataFrame([{"member": m, **G.nodes[m]} for m in members]).set_index("member")
mem_df["yeas"] = [G.degree(m) for m in mem_df.index]
for b in bill_df.index:
    pc = pd.Series([G.nodes[u]["party"] for u in G.neighbors(b)]).value_counts()
    for p in ["D", "R"]:
        bill_df.loc[b, f"yea_{p}"] = pc.get(p, 0)

print(f"{len(members)} members, {len(bills)} bills, {G.number_of_edges()} Yea edges")
print(f"density (edges / possible edges): {G.number_of_edges() / (len(members) * len(bills)):.3f}")
print(f"connected components: {nx.number_connected_components(G)}")
print(f"mean Yeas per member: {mem_df.yeas.mean():.1f}   mean Yeas per bill: {bill_df.yea_count.mean():.1f}")
print("\nmembers by party:", mem_df.party.value_counts().to_dict())
print("\nbills by outcome, with average votes:")
summary = bill_df.groupby("outcome").agg(bills=("yea_count", "size"), yea=("yea_count", "mean"),
                                         nay=("nay_count", "mean"), yea_D=("yea_D", "mean"), yea_R=("yea_R", "mean"))
summary.round(1)

In [ ]:
OUTCOME_COLOR = {"enacted": "#2ca02c", "passed_house_not_enacted": "#98df8a", "failed": FAILED_COLOR}
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for o, c in OUTCOME_COLOR.items():
    s = bill_df[bill_df.outcome == o]
    if len(s):
        axes[0].hist(s.yea_count, bins=range(0, 440, 20), alpha=0.7, color=c, label=f"{o} ({len(s)})")
        axes[1].scatter(s.yea_D, s.yea_R, c=c, alpha=0.7, s=20, label=o)
axes[0].set(xlabel="Yea votes on bill", ylabel="bills", title="Yeas per bill"); axes[0].legend()
axes[1].set(xlabel="Democratic Yeas", ylabel="Republican Yeas", title="Party support per bill")
for p in ["D", "R"]:
    axes[2].hist(mem_df[mem_df.party == p].yeas, bins=30, alpha=0.6, color=PARTY_COLOR[p], label=p)
axes[2].set(xlabel="bills voted Yea", ylabel="members", title="Yeas per member"); axes[2].legend()
plt.tight_layout(); plt.show()

### Bill text
`build_graph.py` also saves the text of every bill.

In [ ]:
# Bill text is stored in data/<congress>/text/<bill>.txt; the first line says which version it is.
from pathlib import Path
def bill_text(b):
    f = Path(f"data/{CONGRESS}/text/{b}.txt")
    return f.read_text(encoding="utf-8") if f.exists() else ""

bill_df["words"] = [len(bill_text(b).split()) for b in bill_df.index]
print(f"text available for {(bill_df.words > 0).sum()} of {len(bill_df)} bills")
print(bill_df.groupby("outcome").words.describe()[["count", "mean", "50%", "max"]].round(0))
shortest = bill_df[bill_df.words > 0].words.idxmin()
print(f"\nshortest bill: {shortest} ({bill_df.words[shortest]} words), first 600 characters:\n")
print(bill_text(shortest)[:600])

## 2. Community detection
Two members are close if they voted Yea on many of the same bills. We project onto members, then run Louvain (finds its own number of groups) and a 2-way spectral split (normalised Laplacian), and compare with party. ARI: 1 = identical to party, ~0 = unrelated.

In [ ]:
MIN_BILLS = 5                       # ignore members with fewer Yeas than this
idx = {m: i for i, m in enumerate(members)}
B = np.zeros((len(members), len(bills)))
for j, b in enumerate(bills):
    for u in G.neighbors(b):
        B[idx[u], j] = 1
keep = B.sum(1) >= MIN_BILLS
mem = [m for m, k in zip(members, keep) if k]; B = B[keep]
W = B @ B.T; np.fill_diagonal(W, 0)                     # co-Yea counts
P = nx.relabel_nodes(nx.from_numpy_array(W), dict(enumerate(mem)))

d = W.sum(1)
v = np.linalg.eigh(np.eye(len(d)) - W / np.sqrt(np.outer(d, d)))[1][:, 1] / np.sqrt(d)
louv = nx.community.louvain_communities(P, weight="weight", seed=236)
lab = {m: i for i, c in enumerate(louv) for m in c}
df = pd.DataFrame({"party": [G.nodes[m]["party"] for m in mem], "spectral": (v > 0).astype(int),
                   "louvain": [lab[m] for m in mem]}, index=mem)

def ari(a, b):
    t = pd.crosstab(a, b).values
    s = comb(t, 2).sum(); sa, sb = comb(t.sum(1), 2).sum(), comb(t.sum(0), 2).sum()
    e = sa * sb / comb(t.sum(), 2)
    return (s - e) / (0.5 * (sa + sb) - e)

print(f"{len(mem)} members used; Louvain found {len(louv)} communities")
print(f"ARI vs party   Louvain: {ari(df.party, df.louvain):.3f}   spectral: {ari(df.party, df.spectral):.3f}\n")
print(pd.crosstab(df.louvain, df.party), "\n")
print("members whose Louvain community has a different majority party than theirs:")
maj = df.groupby("louvain").party.agg(lambda s: s.value_counts().idxmax())
print(df[df.party != df.louvain.map(maj)].assign(name=lambda x: [G.nodes[m]["name"] for m in x.index]))

pos = nx.spring_layout(P, weight="weight", seed=236, iterations=100)
cmap = plt.get_cmap("tab10")
fig, axes = plt.subplots(1, 2, figsize=(15, 7))
for ax, colors, title in [(axes[0], [PARTY_COLOR[p] for p in df.party], "Colored by party"),
                          (axes[1], [cmap(c % 10) for c in df.louvain], "Colored by Louvain community")]:
    nx.draw_networkx_edges(P, pos, alpha=0.005, width=0.3, ax=ax)
    nx.draw_networkx_nodes(P, pos, nodelist=mem, node_color=colors, node_size=25, ax=ax)
    ax.set_title(title); ax.set_axis_off()
plt.show()

## 3. Induced subgraph on 5 bills
Pick 5 bills (a mix of passed and failed), keep them and every member who voted Yea on at least one. Bills are squares, green if they passed the House and red if they failed; members are blue circles. Change `SEED` to resample.

In [ ]:
N_BILLS, N_FAILED, SEED = 5, 2, 236
rng = np.random.default_rng(SEED)
failed = [b for b in bills if G.nodes[b]["outcome"] == "failed"]
passed = [b for b in bills if G.nodes[b]["outcome"] != "failed"]
n_f = min(N_FAILED, len(failed))
sample = list(rng.choice(failed, n_f, replace=False)) + list(rng.choice(passed, N_BILLS - n_f, replace=False))
S = G.subgraph(sample + [m for b in sample for m in G.neighbors(b)]).copy()
print(S.number_of_nodes(), "nodes,", S.number_of_edges(), "edges")
print(pd.DataFrame([{"bill": b, "outcome": G.nodes[b]["outcome"], "yea": G.nodes[b]["yea_count"],
                     "nay": G.nodes[b]["nay_count"]} for b in sample]).to_string(index=False))

ang = np.linspace(0, 2 * np.pi, len(sample), endpoint=False)
p = nx.spring_layout(S, pos={b: (2 * np.cos(a), 2 * np.sin(a)) for b, a in zip(sample, ang)},
                     fixed=sample, k=0.6, iterations=200, seed=SEED)
mem_nodes = [n for n in S if S.nodes[n]["kind"] == "member"]
fig, ax = plt.subplots(figsize=(10, 10))
nx.draw_networkx_edges(S, p, alpha=0.05, width=0.4, ax=ax)
nx.draw_networkx_nodes(S, p, nodelist=mem_nodes, node_color=MEMBER_COLOR, node_size=18, alpha=0.8, ax=ax)
for flag, color, label in [(False, PASSED_COLOR, "bill: passed House"), (True, FAILED_COLOR, "bill: failed")]:
    nl = [b for b in sample if (G.nodes[b]["outcome"] == "failed") == flag]
    if nl:
        nx.draw_networkx_nodes(S, p, nodelist=nl, node_color=color, node_shape="s", node_size=450, ax=ax, label=label)
nx.draw_networkx_labels(S, p, labels={b: b for b in sample}, font_size=8, ax=ax)
ax.scatter([], [], c=MEMBER_COLOR, s=18, label="House member")
ax.legend(loc="lower left"); ax.set_axis_off(); plt.show()

---
## 4. Two separate graphs: sponsors only vs. Yea votes only
Same node types (House members, bills), different edges:
- **Yea graph** `G_yea`: member -- bill if the member voted Yea on the bill's final House vote (this is `G` from above).
- **Sponsor graph** `G_spon`: member -- bill if the member is the bill's primary sponsor (from `sponsors.csv`). Senate-originated bills have a senator as sponsor, so they have no House member to link to and stay as isolated bills.

In [ ]:
G_yea = G
sp = pd.read_csv(f"data/{CONGRESS}/sponsors.csv")
G_spon = nx.Graph()
G_spon.add_nodes_from((n, d) for n, d in G.nodes(data=True))          # same nodes and attributes
G_spon.add_edges_from((r.bioguide, r.bill) for r in sp.itertuples() if r.bioguide in G_spon and r.bill in G_spon)

def graph_stats(H, name):
    deg_m = [H.degree(m) for m in members]; deg_b = [H.degree(b) for b in bills]
    comps = [c for c in nx.connected_components(H) if len(c) > 1]
    return {"graph": name, "edges": H.number_of_edges(),
            "members with >=1 edge": sum(d > 0 for d in deg_m), "bills with >=1 edge": sum(d > 0 for d in deg_b),
            "mean member degree": np.mean(deg_m), "max member degree": max(deg_m),
            "mean bill degree": np.mean(deg_b), "non-trivial components": len(comps),
            "density": H.number_of_edges() / (len(members) * len(bills))}

pd.DataFrame([graph_stats(G_yea, "Yea votes"), graph_stats(G_spon, "Sponsors")]).set_index("graph").T.round(3)

In [ ]:
print(f"sponsor graph: {sum(1 for b in bills if G_spon.degree(b) == 0)} of {len(bills)} bills have no House sponsor in the graph (Senate bills or missing sponsor)")
spon_by_outcome = pd.Series({b: G_spon.degree(b) > 0 for b in bills}).groupby(
    [G.nodes[b]["outcome"] for b in bills]).agg(["sum", "size"]).rename(columns={"sum": "with House sponsor", "size": "bills"})
print(spon_by_outcome, "\n")

# Who sponsors the most of these bills, and does it differ by party?
top = sorted(members, key=G_spon.degree, reverse=True)[:10]
print(pd.DataFrame({"name": [G.nodes[m]["name"] for m in top], "party": [G.nodes[m]["party"] for m in top],
                    "bills sponsored": [G_spon.degree(m) for m in top]}).to_string(index=False))
print("\nbills sponsored per party:", pd.Series({m: G_spon.degree(m) for m in members}).groupby(
    {m: G.nodes[m]["party"] for m in members}).sum().to_dict())

In [ ]:
# Side by side: degree distributions of members in each graph
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, H, title in [(axes[0], G_yea, "Yea graph: Yeas per member"), (axes[1], G_spon, "Sponsor graph: bills sponsored per member")]:
    for p in ["D", "R"]:
        ax.hist([H.degree(m) for m in members if G.nodes[m]["party"] == p], bins=30, alpha=0.6, color=PARTY_COLOR[p], label=p)
    ax.set(title=title, xlabel="degree", ylabel="members"); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Community detection on each graph, using the same method as section 2.
def communities(H, min_bills=1):
    """Project H onto members (shared bills), then Louvain; returns labels for members with enough edges."""
    ids = {m: i for i, m in enumerate(members)}
    M = np.zeros((len(members), len(bills)))
    for j, b in enumerate(bills):
        for u in H.neighbors(b):
            M[ids[u], j] = 1
    keep = M.sum(1) >= min_bills
    mem_k = [m for m, k in zip(members, keep) if k]; M = M[keep]
    W = M @ M.T; np.fill_diagonal(W, 0)
    Pk = nx.relabel_nodes(nx.from_numpy_array(W), dict(enumerate(mem_k)))
    comms = nx.community.louvain_communities(Pk, weight="weight", seed=236)
    lab = {m: i for i, c in enumerate(comms) for m in c}
    df_k = pd.DataFrame({"party": [G.nodes[m]["party"] for m in mem_k], "louvain": [lab[m] for m in mem_k]}, index=mem_k)
    return df_k, Pk, W.sum()

rows = []
for name, H, mb in [("Yea votes", G_yea, MIN_BILLS), ("Sponsors", G_spon, 1)]:
    df_k, Pk, total_w = communities(H, mb)
    rows.append({"graph": name, "members used": len(df_k), "member-member edges": Pk.number_of_edges(),
                 "communities": df_k.louvain.nunique(),
                 "ARI vs party": ari(df_k.party, df_k.louvain) if df_k.louvain.nunique() > 1 else float("nan")})
pd.DataFrame(rows).set_index("graph")

**Reading the result.** Every bill has exactly one primary sponsor, so in the sponsor graph two members are never linked through a shared bill: the graph is a set of stars (one member plus the bills they sponsored). Projecting onto members gives no member-member edges, so community detection has nothing to work with. To get structure from sponsorship we would need **cosponsors** (many members per bill), which `build_graph.py` does not fetch yet.

In [ ]:
# Picture of the sponsor graph: members who sponsored at least one bill (blue) with their bills (green passed / red failed)
active = [m for m in members if G_spon.degree(m) > 0]
H = G_spon.subgraph(active + [b for m in active for b in G_spon.neighbors(m)])
p = nx.spring_layout(H, k=0.15, seed=236, iterations=100)
fig, ax = plt.subplots(figsize=(11, 11))
nx.draw_networkx_edges(H, p, alpha=0.3, width=0.5, ax=ax)
nx.draw_networkx_nodes(H, p, nodelist=[n for n in H if G.nodes[n]["kind"] == "member"],
                       node_color=[PARTY_COLOR[G.nodes[n]["party"]] for n in H if G.nodes[n]["kind"] == "member"],
                       node_size=[20 + 25 * H.degree(n) for n in H if G.nodes[n]["kind"] == "member"], ax=ax)
for flag, color, label in [(False, PASSED_COLOR, "bill: passed House"), (True, FAILED_COLOR, "bill: failed")]:
    nl = [n for n in H if G.nodes[n]["kind"] == "bill" and (G.nodes[n]["outcome"] == "failed") == flag]
    nx.draw_networkx_nodes(H, p, nodelist=nl, node_color=color, node_shape="s", node_size=12, ax=ax, label=label)
ax.set_title("Sponsor graph (member color = party, size = bills sponsored)"); ax.legend(); ax.set_axis_off(); plt.show()